# LAB 3A: Single-Layer Perceptron

## Overview
This notebook is organized into clear labeled sections to make the code easier to follow without changing the original logic.

### Section guide
1. Imports and setup
2. Perceptron definition
3. Training and evaluation
4. Decision boundary plots


In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Part A: Perceptron definition


In [ ]:
def perceptron(x, w, b):
    return 1 if np.dot(w, x) + b >= 0 else 0

gates = {
    "AND":    ([ 1,  1], -1.5),
    "OR":     ([ 1,  1], -0.5),
    "NOR":    ([-1, -1],  0.5),
    "NAND":   ([-1, -1],  1.5),
    "ANDNOT": ([ 1, -1], -0.5)
}
inputs = [(0, 0), (0, 1), (1, 0), (1, 1)]

print("--- Part A: Manual Perceptron Weights ---")
for name, (w, b) in gates.items():
    print(f"{name} gate (w = {w}, bias = {b})")
    for a, c in inputs:
        print(f"  {a}, {c} -> {perceptron([a, c], w, b)}")

print("NOT gate (w = [-1], bias = 0.5)")
for a in (0, 1):
    print(f"  {a} -> {perceptron([a], [-1], 0.5)}")
print("\n" + "="*50 + "\n")


## Part B: Training and evaluation


## Plot decision boundaries


In [ ]:
def step(z):
    return np.where(z > 0, 1, 0)

def train_perceptron(X, t, lr=0.1, max_epochs=100):
    w = np.zeros(X.shape[1])
    b = 0.0
    errors_per_epoch = []
    for epoch in range(1, max_epochs + 1):
        errors = 0
        for x_i, t_i in zip(X, t):
            y_i = step(np.dot(w, x_i) + b)
            if y_i != t_i:
                w += lr * (t_i - y_i) * x_i
                b += lr * (t_i - y_i)
                errors += 1
        errors_per_epoch.append(errors)
        if errors == 0:
            return w, b, epoch, errors_per_epoch
    return w, b, max_epochs, errors_per_epoch

X2 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
datasets = {
    "AND":  (X2, np.array([0, 0, 0, 1])),
    "OR":   (X2, np.array([0, 1, 1, 1])),
    "NOR":  (X2, np.array([1, 0, 0, 0])),
    "NAND": (X2, np.array([1, 1, 1, 0])),
    "NOT":  (np.array([[0], [1]]), np.array([1, 0])),
    "XOR":  (X2, np.array([0, 1, 1, 0])),
}

results = {}
print(f"{'Gate':5s} {'epochs':>7s} {'weights':>16s} {'bias':>6s} {'predictions':>14s} {'accuracy':>8s}")
for name, (X, t) in datasets.items():
    w, b, ep, errs = train_perceptron(X, t)
    pred = step(X @ w + b)
    acc = (pred == t).mean() * 100
    results[name] = (w, b, errs)
    status = f"{ep}" if errs[-1] == 0 else f"{ep}*"
    print(f"{name:5s} {status:>7s} {str(np.round(w,2)):>16s} {b:6.2f} {str(pred):>14s} {acc:7.0f}%")

print("* = did not converge within 100 epochs")
print("\nXOR: misclassified samples per epoch (first 12 epochs):", results["XOR"][2][:12])
print("XOR: misclassified samples in the last epoch:", results["XOR"][2][-1])

In [ ]:
# ---- Plot Decision Boundaries ----
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
for ax, name in zip(axes, ["AND", "OR", "XOR"]):
    X, t = datasets[name]
    w, b, _ = results[name]
    xs = np.linspace(-0.5, 1.5, 100)
    if abs(w[1]) > 1e-9:
        ax.plot(xs, -(w[0] * xs + b) / w[1], "k--", label="decision boundary")
    ax.scatter(X[t == 0, 0], X[t == 0, 1], c="tab:red", s=120, label="class 0", edgecolors="k")
    ax.scatter(X[t == 1, 0], X[t == 1, 1], c="tab:green", s=120, label="class 1", edgecolors="k")
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.grid(True)
    ax.set_title(f"{name}: " + ("converged" if results[name][2][-1] == 0 else "FAILS (not linearly separable)"))
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.show()